# பாடம் 13 - Cognee அறிவு வரைபடங்களுடன் முகவர் நினைவகம்


## அமைப்பு

இந்த நோட்புக் [**Cognee**](https://www.cognee.ai/) அறிவு வரைபடங்களையும் **Microsoft Agent Framework** (MAF) ஐப் பயன்படுத்தி நிலைத்த நினைவகத்துடன் கூடிய **கோடிங் உதவியாளரை** எப்படி கட்டமைப்பது என்பதை விளக்குகிறது.

Cognee அமைக்கப்படாத உரையை ஒரு கட்டமைக்கப்பட்ட, விசாரணை செய்யக்கூடிய அறிவு வரைபடமாக மாற்றுகிறது, அதற்கு பின் வெக்டர் இணைப்புகள் ஆதரவாக இருக்கும் — உங்கள் உதவியாளருக்கு ஒரு விரிவான, தொடர்பு தெரிந்த நீண்டகால நினைவகத்தை வழங்குகிறது.

### நீங்கள் கற்றுக்கொள்ளவிருக்கும் விஷயங்கள்
1. **அறிவு வரைபடங்களை கட்டமைக்கவும்**: டெவலப்பர் சுயவிவரங்கள் மற்றும் சிறந்த நடைமுறைகளை கட்டமைக்கப்பட்ட, விசாரணைக்கூடிய அறிவாக மாற்றவும்.
2. **Cognee ஐ MAF உடன் ஒருமைப்படுத்தவும்**: MAF உதவியாளர் Cognee அறிவு வரைபடத்தை விசாரிக்க `@tool` செயல்பாடுகளைப் பயன்படுத்தவும்.
3. **அமர்வு தெரிந்த உரையாடல்கள்**: ஒரே அமர்வின் பல கேள்விகளில் உள்ள சூழலை பராமரிக்கவும்.
4. **நீண்டகால நினைவகம்**: முக்கிய அறிவை அமர்வுகளுக்கு மத்தியில் நிலைத்திருக்கவைத்து அதனை புதிய உரையாடலில் மீட்டெடுக்கவும்.

### முன்-தேவைகள்
- Python 3.9+
- அமர்வு மேலாண்மைக்கான உள்ளூர் ரெடிஸ் இயக்கத்துடன் (`docker run -d -p 6379:6379 redis`)
- ஒரு LLM API விசை (உதாரணமாக OpenAI) — `.env` இல் `LLM_API_KEY` அமைக்கவும்
- `.env` இல் `CACHING=true` (Cognee அமர்வுகளுக்கான தேவையானது)
- ஒரு Microsoft Foundry திட்டம் மற்றும் அதில் chat மாடல் நிறுவப்பட்டது
- `.env` இல் `AZURE_AI_PROJECT_ENDPOINT` மற்றும் `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI ஐ உறுதிபடுத்தியது (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## முகவர் நினைவக வகைகள்

இந்த நோட்புக் முக்கிய பாடம் 13 நோட்புக்கில் உள்ள அதே மூன்று நினைவக வகைகளை ஆராய்கிறது, ஆனால் Cognee ஐ நீண்டகால நினைவக பேக் எண்ட் ஆக பயன்படுத்துகிறது:

| நினைவக வகை | செயல்திட்டம் | ஆயுள் காலம் |
|---|---|---|
| **செயற்பாட்டுப்** | `agent.create_session()` (MAF) | ஒரே உரையாடல் |
| **குறுகியகால** | Cognee session cache (Redis) | ஒரே அமர்வு |
| **நீண்டகால** | Cognee knowledge graph + vectors | நிரந்தர |

### Cognee இன் நினைவக கட்டமைப்பு
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## கோக்னி சேமிப்பை தயாரிக்கவும்


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## பகுதி 1 — அறிவுத்தளத்தை உருவாக்குதல்

நமது குறியீட்டு உதவியாளருக்கான பரந்த அறிவுத்தளத்தை உருவாக்க, நாம் மூன்று வகையான தரவுகளை சீராக எடுக்கும்:

1. **உறுப்பாளர் சுயவிவரம்** — தனிப்பட்ட நிபுணத்துவம் மற்றும் தொழில்நுட்ப பின்னணி
2. **பைதான் சிறந்த நடைமுறைகள்** — நடைமுறை வழிகாட்டல்களுடன் பைதானின் சென்
3. **வரலாற்று உரையாடல்கள்** — முன்னைய கேள்வி & பதில் அமர்வுகள், உறுப்பினர்களும் AI உதவியாளர்களும் இடையே


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## அறிவு வரைபடத்தை காட்சிப்படுத்தவும்

Cognee அதன் கண்டுபிடித்த அங்கங்கள் மற்றும் உறவுகளின் ஒரு இடையூறு HTML காட்சிப்படுத்தலை உருவாக்க முடியும்.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify உடன் நினைவாற்றலை வளப்படுத்து

`memify()` அறிவுக் காட்சியை பகுக்கி, அதிந்த அறிவு விதிகள் உருவாக்குகிறது — முறைமைகள், சிறந்த நடைமுறைகள் மற்றும் கருத்துகளுக்கிடையேயான தொடர்புகளை கண்டறிந்து.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## பகுதி 2 — Cognee கருவிகளுடன் MAF முகவர்

இப்போது நாங்கள் `@tool` செயல்களைப் பயன்படுத்தி Cognee அறிவு குறிச்சொல் மூலம் கேள்வி கேட்கக்கூடிய MAF முகவரை உருவாக்குகிறோம். இது முகவருக்கு உரையாடல் சூழலைச் சேமிக்கும் போது முழு வளைவு-அறிமுக அமைதிச் سرچின் முழு சக்தியையும் பயன்படுத்த அனுமதிக்கிறது.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## அம்மொழிக் கையாளல் அமையுடன் வேலை நினைவகம்

`AgentSession` (`agent.create_session()` மூலமாக உருவாக்கப்பட்டது) அமையத்தின் உள்ளே வேலை நினைவகத்தை வழங்குகிறது. முகவர் முன் செய்திகளை நினைவில் வைக்கவோ, அதே சமயம் Cognee இன் நீண்டகால அறிவுப் பிணையத்தை விசாரிக்கவோ முடியும்.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## புதிய அமர்வு — நீண்ட கால நினைவுகள் நிலைத்திருக்கின்றன

புதிய அமர்வைத் தொடங்குவது செயற்கையான நினைவகத்தை அழிக்கிறது, ஆனால் Cognee அறிவுக் குழு இன்னும் கிடைக்கும். முகவர் முழுமையாக புதிய உரையாடலில் அதே நீண்ட கால அறிவை மீட்டெடுக்க முடியும்.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## சுருக்கம்

இந்த நோட்புக்கில் நீங்கள் **MAF இன் செயல்பாட்டு நினைவகம்** (`agent.create_session()`) மற்றும் **Cognee இன் நீண்டகால அறிவுக் கூட்டு வரைபடத்தை** இணைக்கும் ஒரு குறியீடு உதவியாளரை உருவாக்கினீர்கள்.

### நீங்கள் கற்றுக்கொண்டது என்ன
1. **அறிவுக் கூட்டு வரைபட கட்டமைப்பு**: Cognee அமைப்பற்ற உரையை எடுத்துக்கொண்டு ஒரு கூட்டு + வெக்டர் நினைவகத்தை உருவாக்குகிறது.
2. **memify மூலம் கூட்டு விருத்தி**: தற்போதைய கூட்டின் மேல் பெறப்பட்ட உண்மைகள் மற்றும் செறிவுற்ற உறவுகள்.
3. **MAF + Cognee ஒருங்கிணைப்பு**: `@tool` செயல்பாடுகள் MAF முகவரிக்கு Cognee இன் கூட்டைக் இயற்கையாகக் கேட்க அனுமதிக்கிறது.
4. **செயல்பாட்டு நினைவகம் + நீண்டகால நினைவகம்**: `AgentSession` (`agent.create_session()` மூலம்) அமர்வு சூழலை வழங்குகிறது, Cognee স্থாயி அறிவை வழங்குகிறது.
5. **NodeSets உடன் வடிகட்டப்பட்ட தேடல்**: அறிவுக் கூட்டு வரைபாட்டின் குறிப்பிட்ட துணைத்தொகுதிகளை இலக்கு வைக்க (எ.கா., கொள்கைகள் மட்டும்).

### முக்கிய எடுத்துக்காட்டுக்கள்
- **Cognee** கச்சா உரையை கட்டமைக்கப்பட்ட, உறவுகளை உணர்ந்த நினைவாக மாற்றுகிறது — ஒரு சமமில்லாத வெக்டர் சேமிப்பை விட பலமானது.
- **`@tool` செயல்பாடுகள்** MAF முகவர்களுக்கும் வெளி அறிவுக் கணினி அமைப்புகளுக்குமான பாலமாகச் செயல்படுகின்றன.
- **`AgentSession`** (`agent.create_session()` மூலம்) ஒவ்வொரு உரையாடல் சூழ்லையும் நீண்டகால அறிவிலிருந்து தனித்தனியாக வைக்கிறது.
- அதே அறிவுக் கூட்டு பல அமர்வுகளுக்கும் முகவர்களுக்கும் சேவை செய்கிறது.

### உலகளாவிய பயன்பாடுகள்
- **முன்னணிக் கம்ப்யூட்டர் உதவியாளர்கள்**: குறியீட்டு மதிப்பாய்வு, சம்பவ பகுப்பாய்வு, கட்டமைப்பு உதவியாளர்கள்
- **வாடிக்கையாளர்-நோக்கிய முனை உதவியாளர்கள்**: தயாரிப்பு ஆவணங்கள், கேள்வி பதில்கள் மற்றும் CRM குறிமேடுகளுக்கு ஆதரவுத் தரும் முகவர்கள்
- **உள்நாட்டு நிபுணர் உதவியாளர்கள்**: கொள்கை, சட்ட மற்றும் பாதுகாப்பு உதவியாளர்கள் வழிகாட்டல்களைப் பற்றிய வாதமிடுதல்
- **ஒற்றை தரவு அடுக்கு**: கட்டமைக்கப்பட்ட மற்றும் அமைக்கப்படாத தரவுகளை ஒன்று சேர்க்கும் கூட்டு தேடக்கூடிய வரைபடம்

### அடுத்த படிகள்
- Cognee இல் காலமாற்ற உணர்வு முயற்சி செய்ய
- துறைசாரா கூட்டு தர தர முடிச்சுக்கான OWL ஆன்டாலஜி வரையறுத்தல்
- காலத்தின்போக்கு மேம்படுத்த பயன்படுத்துபவர் பின்னூட்ட முறைகள் சேர்க்க
- ஒரே Cognee நினைவக அடுக்கை பகிரும் பல முகவர் அமைப்புகளுக்காக பரிமாணத்தை விரிவாக்குக


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
